# EA-LSTM: Daymet + PCA10 mean/std, H256, dropout 0.2, adaptive max50

## Two reusable input files
Keep this notebook and `daymet_pca10_discharge_1980_2009.csv` together. Select **Model A PyTorch (GPU)** and run cells in order. The CSV is the sole model dataset. The completed H256 LSTM seed-42 benchmark and its preprocessing reference are embedded in this notebook for portable comparison. All outputs and protocol metadata are generated by this notebook under `outputs/`; no other experiment files are required to rerun. Copy these two inputs into an empty folder to train afresh; rerunning in the same folder resumes a matching epoch checkpoint.

## Controlled comparison with the new H256 LSTM
Same 1,326 training donors and 100 unseen IDs; train 1980–1999, validate 2000–2004, test 2005–2009. Same seven Daymet meteorological variables and 20 static PCA descriptors (PC01–PC10 means and standard deviations), donor-only scaling, loss, 365-day windows, physical training batch 128, validation/inference batch 256, 256 sampled windows per donor per epoch, validation cap 128 per donor, Adam, gradient clipping 1.0, seed 42, one recurrent layer with 256 memory cells, output dropout 0.2 and linear head. Same maximum 50 epochs and validation-only checkpoint selection. LR starts at 0.001 and halves after three consecutive non-improving validation epochs, minimum 0.00001. Stop after ten consecutive non-improving epochs. Improvement means strictly lower validation loss. PCA is not refitted. No extra forget-bias boost or orthogonal/recurrent-identity initialization is added.

## What makes this an EA-LSTM
The input gate is **i = sigmoid(W_static × attributes + b_i)** and is constant through the window. Forget, candidate and output gates receive **meteorology and recurrent hidden state only**, with no direct static-input weights. Memory updates use **c_t = f_t × c_(t−1) + i × g_t**; hidden state uses **h_t = o_t × tanh(c_t)**. Static properties influence later gates indirectly through the hidden state, as in the EA equations.

To retain fast cuDNN execution, this notebook evaluates those equations using a dense PyTorch LSTM with permanent structural zeros: meteorology-to-input-gate weights, hidden-to-input-gate weights and static-to-other-gate weights are zero. Gradient masks keep them zero during Adam updates, and checkpoints assert those constraints. Both PyTorch bias vectors are retained; their sum is the effective gate bias. The same uniform initialization policy as the H256 baseline is used. There are **292,097 stored parameter slots, of which 82,688 are structural zeros and 209,409 are active/unmasked trainable slots**. The models have equal hidden size, not equal active parameter count. Direct independent EA recurrence tests verify both forward values and gradients on CPU and CUDA, plus gate separation and zero preservation after optimization.

References: [EA-LSTM paper, Kratzert et al. (2019)](https://hess.copernicus.org/articles/23/5089/2019/index.html); [NeuralHydrology EA-LSTM equations and gate separation](https://neuralhydrology.readthedocs.io/en/latest/api/neuralhydrology.modelzoo.ealstm.html). This is a standalone mathematically equivalent implementation; the NeuralHydrology package is not required.

## Testing, flow errors and limitations
Test results are computed only after validation-best checkpoint selection. Observed test Q10 and Q90 define within-basin low-flow (Q ≤ Q10) and high-flow (Q ≥ Q90) days, inclusive of ties. Quantiles use finite observed/predicted pairs in 2005–2009, only for reporting; they do not enter training or scaling. Thresholds and test dates must match the H256 baseline. Report subset counts, RMSE, MAE, mean bias and volume bias; zero observed subset volume gives undefined percentage bias. Missing discharge remains missing. Negative predictions are not clipped, preserving the baseline's handling.

Save and compare all 1,426 basin rows, seen/unseen separately, NSE/KGE negative counts, low/high-flow errors, daily predictions, training history and best epoch. Epoch checkpoints include optimizer, both stopping counters and Python/NumPy/PyTorch/CUDA RNG states. One seed tests this architectural change under the same training protocol; it does not establish repeatability. Original basin-overlap limitations and repeated inspection of the unseen set remain unchanged. GPU arithmetic may vary slightly across hardware/software.


In [1]:
from pathlib import Path
import csv,json,time,random,hashlib,base64,zlib,math,os
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader,Sampler
from IPython.display import display,HTML
CSV_PATH=Path.cwd()/'daymet_pca10_discharge_1980_2009.csv'
RUN_NAME='results_seed42'
OUTPUT_BASE=CSV_PATH.parent/'outputs'
OUTPUT_BASE.mkdir(exist_ok=True)
OUT=OUTPUT_BASE/RUN_NAME
OUT.mkdir(exist_ok=True)
FEATURES=['prcp','tmax','tmin','srad','vp','dayl','pet']
STATIC_FEATURES=['PC01_mean', 'PC01_std', 'PC02_mean', 'PC02_std', 'PC03_mean', 'PC03_std', 'PC04_mean', 'PC04_std', 'PC05_mean', 'PC05_std', 'PC06_mean', 'PC06_std', 'PC07_mean', 'PC07_std', 'PC08_mean', 'PC08_std', 'PC09_mean', 'PC09_std', 'PC10_mean', 'PC10_std']
CFG=dict(seed=42,epochs=50,batch=128,windows_per_basin=256,validation_cap=128,lookback=365,hidden=256,learning_rate=.001,gradient_clip=1.,train_start='1980-01-01',validation_start='2000-01-01',test_start='2005-01-01',end_exclusive='2010-01-01',features=FEATURES,static_features=STATIC_FEATURES)
assert CSV_PATH.is_file(), 'Put the CSV beside this notebook, or edit CSV_PATH.'
assert torch.cuda.is_available(), 'Select a GPU Jupyter session/kernel.'
torch.set_num_threads(2)
random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark=False
DEVICE=torch.device('cuda:0')
print('GPU:',torch.cuda.get_device_name(0));print('CSV:',CSV_PATH);print('Only model inputs:',FEATURES);print('Static inputs:',STATIC_FEATURES)

CFG.update(output_dropout=0.2,lr_factor=0.5,lr_patience=3,min_lr=0.00001,early_stopping_patience=10,improvement_rule='strictly lower validation loss',flow_thresholds='test observed Q10/Q90, inclusive; reporting only')
import fcntl
if '_run_lock' not in globals():
    _run_lock=(OUT/'run.lock').open('a')
    fcntl.flock(_run_lock.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)

CFG.update(architecture='EA-LSTM masked cuDNN',effective_parameters=209409,structural_zero_parameters=82688,static_input_gate_only=True,initialization='same PyTorch uniform policy and two-bias parameterization as H256 LSTM baseline')


GPU: NVIDIA H100 80GB HBM3 MIG 1g.10gb
CSV: /home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (Daymet + PCA10 EA-LSTM H256 Dropout02 Adaptive50)/daymet_pca10_discharge_1980_2009.csv
Only model inputs: ['prcp', 'tmax', 'tmin', 'srad', 'vp', 'dayl', 'pet']
Static inputs: ['PC01_mean', 'PC01_std', 'PC02_mean', 'PC02_std', 'PC03_mean', 'PC03_std', 'PC04_mean', 'PC04_std', 'PC05_mean', 'PC05_std', 'PC06_mean', 'PC06_std', 'PC07_mean', 'PC07_std', 'PC08_mean', 'PC08_std', 'PC09_mean', 'PC09_std', 'PC10_mean', 'PC10_std']


## 1. Read and verify the unchanged CSV

In [2]:
EXPECTED_SPLIT_HASH="779bde7ce5db72c8158f26f161865c4987eb4ce6a0ac7469383256b41cc2d844"
def atomic(path,obj):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2));tmp.replace(path)
def save_csv(path,rows,headers=None):
    headers=headers or list(rows[0])
    with open(path,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=headers);w.writeheader();w.writerows(rows)

from itertools import groupby
records=[];preview=[];expected_dates=np.arange(np.datetime64('1980-01-01'),np.datetime64('2010-01-01'))
started=time.time()
with open(CSV_PATH,newline='') as f:
    reader=csv.reader(f);header=next(reader)
    expected=['basin_id','split','date',*FEATURES,'q_obs_mm_day',*STATIC_FEATURES,'seen_unseen','n_subbasins','single_subbasin_flag','time_split']
    assert header==expected,header
    for basin,group in groupby(reader,key=lambda row:row[0]):
        chunk=list(group);splits={r[1] for r in chunk};assert len(splits)==1
        split=next(iter(splits));dates=np.array([r[2] for r in chunk],dtype='datetime64[D]');assert np.array_equal(dates,expected_dates),basin
        data=np.array([r[3:11] for r in chunk],dtype='float32');assert data.shape==(len(dates),8)
        static=np.array(chunk[0][11:31],dtype='float64');assert np.isfinite(static).all()
        assert all(r[11:34]==chunk[0][11:34] for r in chunk),'Static feature or basin metadata changes within basin'
        seen='Seen' if split=='training_donor' else 'Unseen';assert chunk[0][31]==seen
        expected_period=np.where(dates<np.datetime64('2000-01-01'),'train',np.where(dates<np.datetime64('2005-01-01'),'validation','test'))
        assert np.array_equal(np.array([r[34] for r in chunk]),expected_period)
        assert np.all(data[np.isfinite(data[:,-1]),-1]>=0)
        records.append(dict(id=basin,split=split,dates=dates,x=data[:,:7].copy(),y=data[:,7].copy(),static=static,n_subbasins=int(chunk[0][32]),single_subbasin_flag=int(chunk[0][33])))
        if not preview:preview=chunk[:5]
        if len(records)%200==0:print('CSV catchments loaded:',len(records),flush=True)
ids=[r['id'] for r in records];assert len(ids)==len(set(ids))==1426
split_text=''.join(r['id']+'|'+r['split']+chr(10) for r in sorted(records,key=lambda r:r['id']))
assert hashlib.sha256(split_text.encode()).hexdigest()==EXPECTED_SPLIT_HASH,'Catchment split differs from original baseline.'
donors=[r for r in records if r['split']=='training_donor'];held=[r for r in records if r['split']=='unseen_test']
assert len(donors)==1326 and len(held)==100 and not {r['id'] for r in donors}&{r['id'] for r in held}
hash_csv=hashlib.sha256()
with open(CSV_PATH,'rb') as f:
    for part in iter(lambda:f.read(8*1024*1024),b''):hash_csv.update(part)
CSV_HASH=hash_csv.hexdigest();OUT.mkdir(exist_ok=True)
DIGEST=hashlib.sha256(json.dumps(dict(config=CFG,csv_sha256=CSV_HASH),sort_keys=True).encode()).hexdigest()
atomic(OUT/'data_audit.json',dict(csv=str(CSV_PATH),rows=sum(len(r['dates']) for r in records),basins=1426,donors=1326,unseen=100,csv_sha256=CSV_HASH,dynamic_inputs=FEATURES,static_inputs=STATIC_FEATURES,metadata_excluded=['basin_id','split','seen_unseen','n_subbasins','single_subbasin_flag','time_split'],loader_seconds=round(time.time()-started)))
display(HTML('<table border="1">'+''.join('<tr>'+''.join('<td>'+str(x)+'</td>' for x in row)+'</tr>' for row in [header]+preview)+'</table>'))
print('CSV is the sole data source. Exact original split verified.')


CSV catchments loaded: 200


CSV catchments loaded: 400


CSV catchments loaded: 600


CSV catchments loaded: 800


CSV catchments loaded: 1000


CSV catchments loaded: 1200


CSV catchments loaded: 1400


basin_id,split,date,prcp,tmax,tmin,srad,vp,dayl,pet,q_obs_mm_day,PC01_mean,PC01_std,PC02_mean,PC02_std,PC03_mean,PC03_std,PC04_mean,PC04_std,PC05_mean,PC05_std,PC06_mean,PC06_std,PC07_mean,PC07_std,PC08_mean,PC08_std,PC09_mean,PC09_std,PC10_mean,PC10_std,seen_unseen,n_subbasins,single_subbasin_flag,time_split
CAN_01AD003,training_donor,1980-01-01,0.630463481,-7.76700449,-15.3141403,128.286835,185.841629,29947.8027,0.0535097606,0.687992156,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-02,0,-6.98227119,-16.2555332,154.34343,171.882034,30004.3125,0.0111347819,0.662272871,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-03,0,-13.8265314,-20.5408897,122.875526,119.336349,30065.4141,0.058900591,0.655843019,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-04,0,-11.6368589,-21.8019562,167.424911,106.858482,30131.0566,0.00925885979,0.635267556,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-05,0,-13.6522999,-22.5217896,154.378876,100.219086,30201.1836,0.029025048,0.612763107,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,Seen,31,0,train


CSV is the sole data source. Exact original split verified.


## 2. Prepare inputs and verify baseline-equivalent scaling

In [3]:
def fit_preprocessing(rs):
    assert len(rs)==1326 and all(r['split']=='training_donor' for r in rs)
    sx=np.zeros(7);sx2=np.zeros(7);nx=0;sy=0.;sy2=0.;ny=0;sigmas=[]
    for r in rs:
        train=r['dates']<np.datetime64(CFG['validation_start'])
        x=r['x'][train&np.isfinite(r['x']).all(1)].astype('float64');y=r['y'][train&np.isfinite(r['y'])].astype('float64')
        assert len(x)>0 and len(y)>0,r['id']
        sx+=x.sum(0);sx2+=(x*x).sum(0);nx+=len(x);sy+=y.sum();sy2+=(y*y).sum();ny+=len(y);sigmas.append(float(y.std()))
    p=dict(xmean=sx/nx,xstd=np.maximum(np.sqrt(np.maximum(sx2/nx-(sx/nx)**2,0)),1e-6),ymean=sy/ny,ystd=max(np.sqrt(max(sy2/ny-(sy/ny)**2,0)),1e-6))
    p['loss_scales']=np.maximum(np.array(sigmas)/p['ystd'],.1);return p
class Windows(Dataset):
    def __init__(self,rs,p,split,cap=None):
        self.rs,self.p,self.split=rs,p,split;self.x=[];self.indices=[]
        for r in rs:
            x=((r['x']-p['xmean'])/p['xstd']).astype('float32');self.x.append(x)
            d=r['dates'];L=CFG['lookback'];t=np.arange(L-1,len(d));bad=np.r_[0,np.cumsum(~np.isfinite(x).all(1))];keep=(bad[t+1]-bad[t+1-L])==0
            if split=='train':keep &= d[t]<np.datetime64(CFG['validation_start'])
            elif split=='validation':keep &= (d[t]>=np.datetime64(CFG['validation_start']))&(d[t]<np.datetime64(CFG['test_start']))
            elif split=='test':keep &= d[t]>=np.datetime64(CFG['test_start'])
            else:raise ValueError(split)
            if split!='test':keep &= np.isfinite(r['y'][t])
            ix=t[keep]
            if cap and len(ix)>cap:ix=ix[np.linspace(0,len(ix)-1,cap).astype(int)]
            self.indices.append(ix.astype('int32'))
        self.lengths=np.array([len(a) for a in self.indices]);self.ends=np.cumsum(self.lengths);self.starts=np.r_[0,self.ends[:-1]]
        if split=='train':assert (self.lengths>0).all()
    def __len__(self):return int(self.ends[-1])
    def __getitem__(self,i):
        bi=int(np.searchsorted(self.ends,i,side='right'));t=int(self.indices[bi][i-self.starts[bi]]);L=CFG['lookback']
        y=(self.rs[bi]['y'][t]-self.p['ymean'])/self.p['ystd'] if self.split!='test' else 0.
        dynamic=self.x[bi][t+1-L:t+1];static=np.broadcast_to(self.rs[bi]['static_scaled'],(L,len(STATIC_FEATURES)))
        inputs=np.concatenate([dynamic,static],axis=1)
        return torch.from_numpy(inputs),torch.tensor(y,dtype=torch.float32),bi,t
class EqualBasinSampler(Sampler):
    def __init__(self,ds,per_basin=256,seed=42):self.ds,self.per_basin,self.seed,self.epoch=ds,per_basin,seed,0
    def __len__(self):return len(self.ds.rs)*self.per_basin
    def __iter__(self):
        rng=np.random.default_rng(self.seed+self.epoch);bs=np.repeat(np.arange(len(self.ds.rs)),self.per_basin);rng.shuffle(bs)
        ix=self.ds.starts[bs]+rng.integers(self.ds.lengths[bs]);return iter(ix.tolist())
p=fit_preprocessing(donors)
static_train=np.stack([r['static'] for r in donors]);p['static_mean']=static_train.mean(0);p['static_std']=np.maximum(static_train.std(0),1e-6)
assert np.isfinite(p['static_mean']).all() and np.isfinite(p['static_std']).all()
for r in records:r['static_scaled']=((r['static']-p['static_mean'])/p['static_std']).astype('float32')

train=Windows(donors,p,'train');val=Windows(donors,p,'validation',cap=128)
sampler=EqualBasinSampler(train);loader=DataLoader(train,batch_size=128,sampler=sampler,num_workers=0,pin_memory=True)
val_loader=DataLoader(val,batch_size=256,num_workers=0,pin_memory=True)
assert len(sampler)==1326*256
# Changing test discharge cannot change test-window dates or model-input tensors.
r=held[0];altered=dict(r,y=np.full_like(r['y'],np.nan));ta=Windows([r],p,'test');tb=Windows([altered],p,'test')
assert np.array_equal(ta.indices[0],tb.indices[0]);assert torch.equal(ta[0][0],tb[0][0]);assert ta[0][1].item()==0.
np.savez(OUT/'preprocessing.npz',**p)
atomic(OUT/'fit_metadata.json',dict(fitted_ids=[r['id'] for r in donors],excluded_ids=[r['id'] for r in held],fit_end_exclusive='2000-01-01',static_features=STATIC_FEATURES,experiment_hash=DIGEST))
save_csv(OUT/'window_counts.csv',[dict(basin_id=r['id'],train_windows=int(n),validation_windows=int(v)) for r,n,v in zip(donors,train.lengths,val.lengths)])
print('Training windows:',len(train),'fixed validation windows:',len(val),'validation donors:',int((val.lengths>0).sum()))

assert train[0][0].shape==(365,27)
assert not {r['id'] for r in held}&{r['id'] for r in train.rs}
print('Static scaler fitted once per donor basin; 27 LSTM input channels.')


_expected_scalers=json.loads('{"xmean": [2.9055279725907406, 11.780602787606476, 0.20089469833976542, 298.84966588989425, 761.9424517408969, 43191.314406607664, 2.107951434402401], "xstd": [6.707291284433858, 12.724785326369203, 11.648552845611587, 147.4226124368205, 571.8555907362005, 9685.135205363773, 1.7480695902312124], "ymean": 1.609671384814403, "ystd": 5.388112200284797, "loss_scales": [0.4161625964974366, 0.5093457068062701, 0.5146129813004272, 0.46819931618136923, 0.44934723523341386, 0.5027634654773845, 0.40857916434353636, 0.4440901139380116, 0.4995745600957485, 1.3733037345944596, 0.45007203330645745, 0.4988152571499939, 0.4578604138725993, 0.5729798041212991, 0.47312947556546664, 0.4117073136206368, 0.56738353140814, 0.5384729595326888, 0.5949786809226051, 0.4172663527229141, 0.44993709901588513, 0.46477663335071145, 0.5784102382570971, 0.47334575439268334, 0.522214218041858, 0.46380950910581675, 0.3994372923700707, 0.8390096914415974, 0.39147856959950794, 0.3543775287492334, 0.33462098392894885, 0.6486415974411442, 0.6940772635148317, 0.5726497730266317, 0.6390860600978588, 0.5765940209241208, 0.43217963966948797, 0.36910724132493233, 0.44821177190556954, 0.4556381270248675, 0.6261614836483413, 0.7979002387480361, 0.9716040503811896, 0.9569662099264491, 0.22559109828721607, 0.34475911615494026, 0.29755635281579573, 0.28814336529412915, 0.23292748461899138, 0.15234860771190617, 0.25891950998419094, 0.27713230839866915, 0.15641742986229032, 0.5638459486912163, 0.27975550138850813, 0.3863024742330061, 0.48658607072648086, 0.2692055807523057, 0.2706167813555749, 0.2689657273980052, 0.21640111231229336, 0.39297782153235483, 0.23864165073215737, 0.3469336874479565, 0.40033773080348967, 0.2953696484617494, 0.37567433148599044, 0.45202485770531536, 0.2939052344060382, 0.2822981980819397, 0.23860125073932115, 0.244315889982905, 0.1619718036528206, 0.1668568214882627, 0.20274986539331324, 0.209463129173618, 0.274799679438057, 0.20773151003274015, 0.2524339774239139, 0.1945537736363039, 0.17656544618766423, 0.21537036385342878, 0.6232427757091846, 0.38746054975233674, 0.3154172522917216, 0.358504494638826, 0.31822794596495974, 0.3589287466735632, 0.2603488447291464, 0.3659715280214854, 0.44115849981998584, 0.4876948902327044, 0.4869647332113864, 0.48697470413325694, 0.44334176044642887, 0.4139093435941473, 0.30773387878182945, 0.4394041336286869, 0.5936596986307445, 0.5241765892568463, 0.1623921240407811, 0.4199619649657315, 0.3314150867804786, 0.4455265954463926, 0.15802957589621233, 0.3669432497560899, 0.4556537537664685, 0.3785242683917217, 0.47363262463946365, 0.38362288840314185, 0.44549422869357613, 0.4326770425127943, 0.2886051740703781, 0.3017004258249289, 0.2809118339403475, 0.35381183551529044, 0.5249889300641333, 0.404282780449312, 0.43867156538004975, 0.29556441298436514, 0.39065919397721083, 0.16481221379114377, 0.35089371410857345, 0.2011592809087293, 0.171980591333059, 0.22973143412524522, 0.2673187929385375, 0.2373378153279257, 0.24607060045050788, 0.20001161385127084, 0.22230511096957312, 0.26894458419066636, 0.27089653613275605, 0.30994861880307367, 0.3865048355836397, 0.3788545867337823, 0.24796180825807565, 0.2225771667378802, 0.35695111008235236, 0.40830382766718004, 0.3850968920903091, 0.43630131655222876, 0.3887259536948647, 0.4182327950513891, 0.39290638364201574, 0.43193382221505544, 0.4053022385818364, 0.39359531664529424, 0.42425638472209604, 3.3667149181619562, 0.7386459775944983, 0.6031178322716789, 0.9701043185138832, 0.6933397187956665, 0.682584929073781, 0.42331434096637466, 0.830589091974108, 0.739627730465525, 0.8422778609842136, 0.9859560247128837, 0.7627735694717269, 0.6766448901683, 0.8891764098756186, 0.5890917086219587, 0.48101108856438, 0.4117164652727502, 0.6893321030586909, 0.3302352313328804, 0.3289524802820245, 0.5773465062128292, 0.3311630798318202, 0.45067623738518936, 0.9289779961207768, 1.7331977017890217, 1.8537699336276863, 0.5509165242489051, 0.8160724477029698, 0.4657972975611408, 16.997101182932617, 0.833704310453786, 1.1087483296759166, 0.8641389477561459, 0.5992780469494311, 0.4272913163461824, 0.5625464690662385, 0.6722991950838885, 0.8040378607976005, 1.1759682513705916, 0.6141544699892996, 0.9421800070570407, 0.8619805600449886, 0.9200769133467962, 0.9893242428283828, 0.4722799719966361, 0.5727721814867403, 0.1, 0.1845446923758351, 0.15210990567690832, 0.1333559659382983, 0.20291019760530082, 0.23968806773033832, 0.12912177414446183, 0.16443251767880804, 0.1, 0.1, 0.21905387730847753, 0.29689276214402904, 0.14324956798924196, 0.3153413928809447, 0.23969820229232972, 0.34267435804987373, 0.30549734914150634, 0.27873090922397664, 0.2574771949016865, 0.3105975070931516, 0.45052835619601417, 0.6354039540981524, 0.1, 0.1, 0.1, 0.6094564349940701, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.4451030820751024, 0.3080331544519545, 0.17260968020130912, 0.4264425880684393, 0.1849650267702638, 0.16235872062764473, 0.13122701375974147, 0.14916276559306085, 0.233518270183473, 0.328520091867985, 0.3603933152172664, 0.18859711250544695, 0.2668944097067886, 0.1, 0.11372414847326771, 0.17484419728772482, 0.2198250035846616, 0.18857499715248055, 0.18589439903207497, 0.17669967958150337, 0.1, 0.16720615960261664, 0.1, 0.1, 0.1, 0.33157920114418316, 0.11832431574612157, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.5003581463585092, 0.5612238651905884, 0.10830775304553317, 0.13030286589808357, 0.21471531808130717, 0.37361606312526296, 0.2482611698876711, 0.2068067080666731, 0.23674285966654776, 0.1, 0.1732939486570332, 0.1, 0.1, 0.1, 0.1, 0.18914889009027133, 0.1, 0.1, 0.1, 7.231773782252268, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.13722579808046095, 0.1, 0.1, 0.1, 0.1, 0.1, 1.053471436143838, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.12385263436559112, 0.1374393380748963, 0.1, 0.2457077759176815, 0.32499045414814864, 0.17952349428228723, 0.10722297632916121, 0.18785967870431902, 0.18205380624009282, 0.1, 0.1, 0.14956271923471234, 0.1, 0.1061669471047507, 0.20252903336050943, 0.1, 0.1, 0.1, 0.1, 0.1, 0.12110137101631377, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1676369209429364, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.13950907376251745, 0.1, 0.1271987185388713, 0.17738888061083785, 0.17255103930527962, 0.1, 0.13804288722183194, 0.1356985964087667, 0.1, 0.9126700570297551, 0.2315843798243345, 0.1, 0.12486914159211032, 0.1647143390998631, 0.1, 0.1, 0.13843769849564563, 0.1, 0.25071953779778094, 0.1, 0.1, 0.10476475102082143, 0.1, 0.11577089594738581, 0.1, 0.1, 0.1, 0.1, 0.1, 0.13589142578449, 0.1, 0.1, 0.1, 0.1, 0.11938570818799908, 0.11012244609060298, 0.11178293507754661, 0.29745835578276136, 0.17436502034834123, 0.20095674281915374, 0.19556581239666965, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.18109942449525587, 0.1, 0.25009210892146994, 0.11896479370463309, 0.3680331074368646, 0.4166717960033967, 1.1754369385628831, 0.17630297358060995, 0.34696318377633945, 0.3000486567084081, 0.36898921442983257, 0.12469736576981703, 0.3385405522910868, 0.1805312516367294, 0.2744346159604858, 0.23756832432255498, 0.16590378980597592, 0.16083695632438938, 0.31689226895622885, 0.18868592335546536, 0.1, 0.1474533473907373, 0.11698520417986312, 0.16276956589974445, 0.10679603402019236, 0.1596296467840328, 0.1, 0.1002364267788693, 0.1, 0.3082898584464485, 0.2581634887516542, 0.18402073619897785, 0.11121646297502999, 0.39962693180371495, 0.24406079139424303, 0.3930594059816679, 0.10257065504004423, 0.1, 0.3028196779181036, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1271582049492156, 0.1, 0.10796376770842424, 0.10047878723807997, 0.1, 0.1, 0.1, 0.1, 0.2971290972899811, 0.39927354709819995, 0.416576985270308, 0.376910535989109, 0.31442983269254177, 0.40950521283160535, 0.43349428818545, 0.23191230678936317, 0.23529511909277337, 0.23111517209689647, 0.2026784357994254, 0.2992009796796258, 0.4876567541072395, 0.3526606192599842, 0.46230141232999383, 0.17827430095279373, 0.263397680261127, 0.1369324597556275, 0.10440942070817096, 0.18679134955137958, 0.1, 0.1, 0.1, 0.1, 0.1, 0.39749671411498655, 0.2184910476834171, 0.33621581468904455, 0.1429585235144103, 0.2538397589059218, 0.13099145951318844, 0.24623561120962048, 0.14569146989230233, 0.17627326206531296, 0.17370516880039993, 0.1, 0.1, 0.22338011536810995, 0.13966531885369834, 0.16872420132852278, 0.1, 0.1, 0.12802487276749197, 0.1, 0.12180398215986966, 0.1, 0.1, 0.1, 0.1, 0.1, 0.10198874600561735, 0.1, 0.1, 0.1, 0.1, 0.1, 0.21548228361964078, 0.2765314657447191, 0.23463853570764442, 0.7768660898128059, 1.3178708634691518, 0.7808516322640999, 0.6261211843845421, 0.5323800248709044, 0.21023206364617894, 0.26962913450064696, 0.2959593437392228, 0.5155353091275667, 0.20196354951539655, 0.7034423414921537, 0.5991426672806658, 1.8001470729830102, 0.8677276346868004, 0.7324105004645799, 0.17820602117414, 1.3878331088940203, 0.990329133687532, 1.0805775935418693, 1.2882549615795371, 0.9798452526108063, 1.3134151291263148, 1.7410307577413044, 2.7370198462241246, 1.7435842225038698, 0.7566334916786135, 0.1, 0.7369824797147522, 1.6346049946103947, 1.2807484800618352, 2.2157430752938145, 0.7556120191804606, 1.5015285169812993, 3.07897378864018, 1.6377082635413531, 0.8335517395481015, 4.7934373155484264, 0.9135783922068803, 1.6895396803949956, 1.281564615875234, 2.4510809810872964, 0.1, 0.1, 0.2656269237131494, 0.1, 0.6267279047132974, 0.48604242500779404, 0.5108774140835742, 0.6991136902204395, 0.6857833019008234, 0.9170262414806585, 0.1801908001817517, 0.3224661092765064, 0.1, 0.5114548400284562, 0.30562201727077243, 0.29352958929255124, 0.16060344948099228, 0.8092520972553277, 0.5780007041676077, 0.1673684834421748, 0.31253591030594097, 0.710290859915375, 0.8240620362230242, 0.3951894852250364, 0.23349302077306439, 0.1, 0.3171505755977563, 0.2743026920132721, 0.5241801376853245, 0.1, 0.19799994976935184, 0.29627398243787145, 0.4556468205315635, 0.2673876124129981, 0.14490343559067945, 0.33154652752681923, 1.7381553320536902, 0.12321468741563334, 0.8148928590076657, 0.8618635195941432, 0.6853582613992047, 1.7778317314021976, 0.8754184280812263, 0.5602459042979326, 0.6150990019320698, 2.2852088152368086, 0.35559245890746816, 0.9280385158415562, 1.5255318937028595, 0.6650063841988809, 2.8613312748603703, 1.931451123842891, 0.26411297844919773, 0.4277983083060864, 0.2961920295391354, 0.5088244428795049, 0.38646234766106297, 0.6526400431440421, 0.8865862963187947, 0.7135848502187938, 0.8985909293178163, 0.24574571686268754, 0.4457149935320913, 0.5723382605488978, 0.3005710984580254, 0.347103833565875, 0.6271445178016605, 0.27311597842027485, 0.6543387137097191, 0.7870291010780865, 0.5930706438821752, 0.6320273955660554, 0.31972642273887614, 0.5140216784962638, 0.22521279678478695, 0.8718733372161147, 0.7877516666849852, 0.5007399399458735, 0.6100144338023762, 0.8625822836294458, 0.36631885044396667, 0.5760172405204828, 0.20109587285334826, 0.34245999314092035, 0.2982420178667259, 0.20712371524235706, 0.19727717381284995, 0.36974933398846477, 0.5031465492841578, 0.691787455340955, 0.10754802234111092, 0.2601850454939608, 0.17862126155782757, 0.4686580043750435, 0.4670451764187376, 0.48612538235090896, 0.3913385636056844, 0.43758655847270456, 0.28400640471246436, 1.2036522879807134, 0.6671994029246177, 0.5584114030931544, 1.2998646650726757, 1.7131608192775802, 0.19161756402913185, 0.2871052833690426, 0.14096082026094747, 0.1, 0.22021518532344703, 0.17473424981151126, 0.15136359493206866, 0.1287303384087315, 0.1, 0.17956880969930591, 0.4274308042637021, 0.39012544610621874, 0.1, 0.4204925873123417, 0.17178933509127273, 0.1577046396907845, 0.1554177894763051, 0.2356428207067664, 0.1, 0.23627492668635042, 0.13888244253151838, 0.11414711942999213, 0.43912886358588255, 0.3029210688890087, 0.19165519478284374, 0.1, 0.1, 0.1291771099045109, 0.24469190058985812, 0.16335077417841293, 0.37656476598834543, 0.6371466095279716, 0.1934104382196054, 0.18451376121593815, 0.19600954428033843, 0.1, 0.1, 0.11643566035585398, 0.3101233136101992, 0.16859879603724393, 0.19391141535759815, 0.23239814963923283, 0.12128766871643235, 0.1, 0.19520755996107766, 0.22950748885908895, 0.20235975097137013, 0.1, 0.10607511483549047, 0.38432522038413414, 0.1, 0.1, 0.109024882886908, 0.1, 0.3519851265942867, 0.4269930023502745, 0.40596748797596977, 0.621419756884818, 0.6076334171380025, 0.6254409807574838, 0.9726160174860142, 0.742351343449722, 0.5639003009431665, 0.48537182997720796, 0.45163360581499407, 0.5500614225845599, 0.45248749584889664, 0.5321320599435337, 0.33388753506644736, 0.35931242545736536, 0.38958226694642034, 0.4032252133519091, 0.6358954558222637, 0.5513293990274085, 0.6249951248409228, 0.6489511746052014, 0.5650075603358832, 0.40483174061018523, 0.6948950361257726, 0.481368706537032, 0.4238526611756647, 0.6588398943131686, 0.2186721800486835, 0.5191015483454108, 0.5809988564155172, 0.4342936886329255, 0.44201091106891577, 0.8228691713782277, 0.8098273127909276, 0.3888754057541018, 0.4029517944019515, 0.4832923959648896, 0.5079630322160608, 0.10431376024199981, 0.15441961603984417, 0.36283975791750667, 0.10813657177844059, 0.19103397566332844, 0.34135425086714827, 0.44576588161230585, 0.468613366295433, 0.4038024854324086, 0.44158597408208644, 0.55797921982105, 0.463370581461276, 0.4720320355786268, 0.4103360595466165, 0.4480086741928425, 0.3922265122481887, 0.5033161915029338, 0.5297665809840203, 0.5162858170447207, 0.6246767615939846, 0.40880698111859237, 0.49081609468707427, 0.45395680926284265, 0.26903366263466505, 0.22631989901207428, 0.29750206199799006, 0.49509840253108606, 0.4630120561922972, 0.7970093505304477, 0.45592003355516997, 0.8961443289630537, 0.5371813104825808, 0.3019205392716918, 0.43857942594713845, 0.412528962225871, 0.3225042311638369, 0.3304844364971239, 0.45424282849030284, 0.3594222336835317, 0.47864196401060355, 0.430549204511628, 0.19761144823703217, 0.2942952844581236, 0.3356871590547127, 0.5601332437205137, 0.41535300222112503, 0.3639215787797341, 0.4707672157594422, 0.42298844403582186, 0.3683728506193929, 0.4113727890569261, 0.48149257022219116, 0.6986655831683716, 0.4573900606020022, 0.36116324087852647, 0.32720336470556804, 0.3339124774152683, 0.30994054039024255, 0.3686405754005836, 0.3183330856765074, 0.5598532199271781, 0.3471385212380536, 0.4458031456425195, 0.46189383023725805, 0.3602093822044479, 0.3245607073793436, 0.3550599214576816, 0.428723574516587, 0.6228511533058044, 0.48748139942686886, 0.3883902415519969, 0.532062785782185, 0.20696072131739968, 0.3422738515747546, 0.5208313075491152, 0.31405509911276797, 0.30822177841324827, 0.2823110471294655, 0.33966935378059226, 0.6262183288429114, 0.5470322185994475, 0.41285595515745743, 0.4674737883992379, 0.4152203127685488, 0.46702612013645906, 0.321762316511473, 0.33634402162106, 0.430265346144809, 0.46906292278463313, 0.3587482487829661, 0.5118505703281544, 0.45686068389748824, 0.44956397758310035, 0.35640887039791125, 0.47647741270675303, 0.30513992761669595, 0.2276409865975506, 0.42088237798861944, 0.1, 0.34945127835131407, 0.4188929944009962, 0.30236318455103933, 0.41441865221874746, 0.3211363725723922, 0.23537797230481333, 0.5362762434781971, 0.5571466717161074, 1.454724458043579, 0.10402509728167886, 0.22060011175213656, 0.20279106981790548, 0.20431216792411735, 0.11077341563008723, 0.5350956155222379, 0.4097857928375288, 0.25340139954278096, 0.44069531461012545, 0.4479726803817984, 0.7546578429706329, 0.3635004296013916, 0.422898767097764, 1.131093326606527, 0.7547326477899696, 0.4793456639539957, 0.530723404557288, 0.5376551450149417, 0.4113777039420032, 0.6271556298347825, 0.6508750382722825, 0.6177430533304858, 0.8714337201354493, 0.6635113667124335, 0.5795325753981215, 0.33142354936869023, 0.5687285390523278, 0.5580441156453902, 0.5564409595444313, 0.5457278621715942, 0.9284055599451426, 0.6580175224109683, 0.3965896011942536, 0.44753556418095225, 0.49123739916151954, 0.6384620580451212, 0.4972453358350927, 0.45902064308261564, 0.3516427103754769, 0.36830446577538856, 0.6357745417971835, 0.6394239131314077, 0.6629812363968302, 0.522457651276632, 0.48340434304282287, 0.4688892129404435, 0.3229323209288626, 0.4321389666084039, 0.43787779199687593, 0.38214015008580093, 0.3157846070209256, 0.40587031421072883, 0.22835413066538426, 0.31492880130562845, 0.6399115927888751, 0.5516773030328342, 0.6947896817102697, 0.7516333481576907, 0.2954100928887294, 0.4834922009096235, 0.5777926964682634, 0.6741897224646097, 0.339134279352754, 0.639683968417387, 0.5047984041110595, 0.6811590264081523, 0.7014971915301071, 0.6520570707224373, 0.48201906837206115, 0.4892813061968381, 0.49451008102835625, 0.61202580911694, 0.6246077931109019, 0.7641402183712094, 0.8048628747826757, 0.546894436558743, 0.4643266289772951, 0.79068474332414, 0.6055537794229726, 0.4138594241537571, 0.7854844597562871, 0.34466293371276224, 0.32502735707742153, 0.6092069371275494, 0.5026873959790934, 0.39965394321914116, 0.42389227229110993, 0.5650509324077737, 0.8350201770972344, 0.7559186138668442, 0.6566660812248908, 0.4242713874909589, 0.2799479274610353, 0.28019260332147633, 0.29530297611496015, 0.41157147664137284, 0.1895657256586179, 0.18123639519805052, 0.1619506272602298, 0.3081700772664693, 0.18729574180273534, 0.14682810835105883, 0.1, 0.1, 0.121356279218202, 0.1, 0.1, 0.1, 0.25884999432483946, 0.1, 0.12076819909577319, 0.2662840293859309, 0.320342224232682, 0.33996668134229546, 0.4502028109880395, 0.6020435358815739, 0.4945001724337555, 0.3902963338576819, 0.3591393908681833, 0.31886301937387546, 0.29266173572084214, 0.5243169637279627, 0.3527388604778614, 0.1, 0.1, 0.1, 0.1, 0.1, 0.10849674649269433, 0.16538722753554053, 0.12207191431251438, 0.38620724284181646, 0.4681010641180135, 0.10987055681164624, 0.19708032721453808, 0.11415856882761324, 0.14264274510332312, 0.23021176563440118, 0.4317303721152255, 0.2588191181288594, 0.2553952615221885, 0.46943226335959387, 0.47439448297715275, 0.49725564256022503, 0.4639424704811697, 0.5334550031760045, 0.5766638940906019, 0.6307640598705775, 0.5940003485897616, 0.5614569116932591, 0.3016914097337963, 0.3101628806918519, 0.36281649633469765, 0.3039252510906677, 0.402992727404139, 0.5436589169141313, 0.6910009245382507, 0.7045088998488095, 0.5427177423031962, 0.11007345778324043, 0.21260456780128323, 0.1, 0.4976458464558322, 0.26178464768961796, 0.38315458826164384, 0.41922681087791663, 0.5787548306988893, 0.34435561314501223, 0.1499643723495181, 0.12525269397886266, 0.22781760014606545, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.11029701503110731, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.13918392223740822, 0.6396893496873285, 0.4857327740070918, 0.5063805243429684, 0.48164135341925024, 0.867629325790142, 0.1, 0.42712965965931543, 0.3605999202847734, 0.3673652494832394, 0.1, 0.10360180688294005, 0.45742029164711856, 0.34874008319309086, 0.45213821942882115, 0.5282102102386638, 0.48224560880263867, 0.40007332411963253, 0.5197169730200116, 0.7635766085382507, 0.6161170597341916, 0.6104432125224352, 0.6957383346038314, 0.43231910594026385, 0.5270997634479391, 0.6096446376226423, 0.665311997187944, 0.30877421267764393, 0.3422272637732255, 0.6318273785484233, 0.3577605357942484, 0.5618371857683067, 0.4140069256756205, 0.31595934140144066, 0.3045474410558602, 0.20919915059204205, 0.37837380761477024, 0.1, 0.39799071640700334, 0.1, 0.1860282674100386, 0.6962804523788207, 0.44229036408511324, 0.912326239786875, 0.3997452180101872, 0.7191119326421227, 0.5268266879114684, 0.1, 0.612175452460057, 0.6406829685336959, 0.7621721066556332, 0.8321316641432348, 0.6775182475970224, 1.0381628486371157, 0.13014591305662787, 0.1, 0.1, 0.31633739246751585, 0.3744949201902968, 1.1586216066909099, 0.9708243938651151, 0.27982511579685526, 1.0478577113326286, 0.5609414311428355, 1.193502194656191, 0.8206885522970886, 0.7922576900453626, 0.5695370684772137, 0.7543147608131481, 0.7643877635273095, 0.7100178835114594, 0.4509758473182289, 0.7089238590011154, 0.8472470756167118, 2.136122686490469, 0.5039498838217457, 0.4751539219500605, 0.1, 0.16024398696234593, 0.17987121511758977, 0.25030275295564575, 0.24602690954876516, 0.22333487594142273, 0.34818648222149257, 0.20567849413839437, 0.13077145459703685, 0.2637178395540567, 0.4191922770016534, 0.5715546413110734, 0.48968753085322386, 0.8665224801015253, 0.2947382792168265, 0.2807830740292374, 0.28810973891589114, 0.2564506321431615, 0.1950214736777475, 0.19366007560660461, 0.15494507274616587, 0.10454175169682749, 0.2564924260714497, 0.4075178136018259, 0.2106779603478826, 0.22242655150346444, 0.15142070936384586, 0.290794357656536, 0.1, 0.20469990281894815, 0.17075793334757378, 0.10470373450739538, 0.6715730222624433, 0.33990703770123504, 0.3531319947299594, 0.2088152065190314, 0.6547208589091831, 0.48028850543269547, 0.6665895928140876, 0.3589852714413089, 0.4930788689939527, 0.20370521673289646, 0.12096635971842465, 0.26904373205677073, 0.1, 0.49222504993947974, 0.1865336414211365, 0.1, 0.1, 0.1, 0.2110470459007099, 0.1, 0.1, 0.5039891493354685, 0.1, 0.17879285377506907, 0.1, 0.1, 0.11764380779956839, 0.2363412362998966, 0.3012005128830252, 0.1985538599002479, 0.47013254229255075, 0.1979021953599708, 0.15920467542031616, 0.3150870727441398, 0.1, 0.1, 0.1, 0.3541521823627141, 0.1519628144750746, 0.10919778845397014, 0.16391405650329174, 0.150678877726934, 0.11302936069358133, 0.1277388986846881, 0.10399337216556848, 0.17151309079833338, 0.12661438100877329, 0.41033589615989163, 0.1, 0.6667812918847846, 0.1, 0.7852193320101069, 1.1456787947571734, 0.1, 0.4472040934586142, 0.4947611480433045, 0.15615771948311616, 0.5502117803057829, 0.242022467085951, 0.24882863743715677, 0.9602508106668614, 0.9479879841187159, 0.1, 0.6363565129686789, 0.30743204598133184, 0.28231714614150616, 0.5064455069539924, 0.10854174618003372, 0.1, 0.6153026348044658, 0.6159905973550701, 0.20407408204162716, 0.1279661786870478, 0.6383806261218063, 0.6207468081449585, 0.568613877552298, 1.0717297075302852, 0.8293627310760697, 1.555861042541966, 1.5151514702630793, 1.094495989838386, 1.2526429431007877, 1.1457809701325143, 0.5995706319501543, 0.7726164234589638, 0.6824711842529563, 2.1008623667211372, 1.344060301309026, 1.5375098911333218, 0.8078166803539205, 1.2165865418496067, 1.7953717806327694, 2.649508214188185, 2.0004926445500755, 2.2144859149635465, 0.3915020882123081, 1.1978257310303457, 1.9052288451978279, 0.4369917909869714, 0.8620025485520447, 0.7397530458937032, 1.2411578845846603, 1.2576423480048469, 1.039569535214523, 1.627644345911417, 1.4198120834036487, 1.165974874630636, 1.0458615576360786, 1.595142337540327, 2.5865494196534633, 1.6024311032332967, 0.9372819146990036, 1.178705758973293, 1.2536735522768554, 0.8970942967729203, 0.5728135472180379, 0.10705736840417326, 0.5564828388725552, 0.8395319915765349, 0.25146630609706916, 0.20193803904921942, 0.2692923512687447, 0.4873087674359999, 0.47346745249901684, 0.46702459052164297, 0.8316674522741676, 0.5829642338503963, 0.5815361978686318, 0.4195068335479172, 0.1, 0.1, 0.385666335632685, 0.3985381940375537, 0.45302819632424324, 0.4103441224229005, 0.5208428544643393, 0.17546810046766217, 0.5088285936883021, 0.3938642419840191, 0.597605302382238, 0.1627129951776929, 1.9462845545285477, 1.6033311346658197, 2.0401657661361203, 1.6735115901593187, 1.3243910302355708, 0.8011648634215939, 0.4796037828751541, 1.3283484389867133, 0.6895689146512797, 1.6455869728583463, 1.1807036461419782, 1.7334229849586054, 1.0325225871815689, 1.3037590011711482, 1.3373847127120992, 1.3935529573542385, 1.0434096360856928, 2.0485616414232477, 1.3999628962453483, 1.1543721268597604, 0.5120797715159892, 0.9857763243832398, 0.9534255662458939, 1.4518753046977857, 0.16498327237017232, 2.5509707162420523], "static_mean": [1.6176100177796398e-16, 0.13897333137136886, 1.1855775285589907e-16, 0.2381229224276496, -1.8302248438096177e-17, 0.11149145285722503, 1.1822284394952647e-16, 0.22524031503076153, -7.342877772219172e-17, 0.11676769546894612, -5.103697756018642e-17, 0.09900390411464699, 1.0842675843812803e-16, 0.10582060197759212, 7.702904846569714e-18, 0.1848250979967018, 2.759858706576676e-17, 0.2106058229909605, -9.310467597158176e-17, 0.1607018728038218], "static_std": [0.8933909011828741, 0.12611714401674384, 0.6664639772702723, 0.3006224066103474, 0.6285306685453489, 0.0946263134062219, 0.4771857233746691, 0.21779491295220454, 0.525170739135402, 0.1171023534051667, 0.3131687990073281, 0.4146409639852452, 0.47585756674680457, 0.08664411217424169, 0.3680773300379937, 0.1436394417898001, 0.31526497154954025, 0.13715048886920508, 0.27621959151369196, 0.15323092255068937]}')
assert set(p)==set(_expected_scalers)
for _k,_v in _expected_scalers.items():assert np.allclose(p[_k],np.asarray(_v),rtol=1e-12,atol=1e-12),_k
atomic(OUTPUT_BASE/'preprocessing_matches_baseline.json',dict(status='passed',all_scalers_and_loss_scales_match=True,reference='H256 mean/std LSTM baseline'))

if not (OUTPUT_BASE/'protocol_verification.json').exists():
    atomic(OUTPUT_BASE/'protocol_verification.json',dict(status='passed',csv_sha256=CSV_HASH,rows=1426*10958,basins=1426,seen=1326,unseen=100,schema_and_split_verified=True,preprocessing_matches_baseline=True,preserved_baseline_sha256=json.loads('{"/home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (Daymet + PCA10 H256 Dropout02 Adaptive50)/Daymet_PCA10_H256_Dropout02_Adaptive50.ipynb": "7d1c0f3ec1346734ee1ae4130cb9044594568d3ff4a9f8192b0e2bc957fc57a2", "/home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (Daymet + PCA10 H256 Dropout02 Adaptive50)/daymet_pca10_discharge_1980_2009.csv": "becf34755e5efdd98c992e1ef33173b69a719146dcd21cc708389daed7db5aa2", "/home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (Daymet + PCA10 H256 Dropout02 Adaptive50)/outputs/basin_model_performance.csv": "6dd880e649848f5660f89bf32b0087b3bbeac2e32f9221367b7a4c0e5b0a7fd5", "/home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (Daymet + PCA10 H256 Dropout02 Adaptive50)/outputs/results_seed42/best.pt": "d120ca84525f5575d01370f680a1d355bf580036c6d7575b417fca934a94da23", "/home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (Daymet + PCA10 H256 Dropout02 Adaptive50)/outputs/results_seed42/history.json": "055bd34c0666c281abc6140dff837d652c800b885cb63be41a0d3f7e92b15410"}')))


Training windows: 8129370 fixed validation windows: 165015 validation donors: 1290
Static scaler fitted once per donor basin; 27 LSTM input channels.


## 3. Define and independently verify the EA-LSTM

In [4]:
class MeteorologyLSTM(nn.Module):
    """EA-LSTM equations evaluated using cuDNN with permanent structural weight masks.

    PyTorch gate order is i,f,g,o. Static descriptors enter i only; meteorology
    enters f,g,o only; recurrent h enters f,g,o only. Both PyTorch bias vectors
    are retained, and their sum is the effective bias in each EA equation.
    """
    def __init__(self,hidden=256):
        super().__init__()
        self.lstm=nn.LSTM(27,hidden,batch_first=True)
        self.output_dropout=nn.Dropout(0.2)
        self.head=nn.Linear(hidden,1)
        ih=torch.ones_like(self.lstm.weight_ih_l0)
        hh=torch.ones_like(self.lstm.weight_hh_l0)
        ih[:hidden,:7]=0       # input gate cannot see meteorology
        ih[hidden:,7:]=0       # forget/candidate/output cannot directly see statics
        hh[:hidden,:]=0        # input gate cannot see recurrent hidden states
        self.register_buffer('input_weight_mask',ih)
        self.register_buffer('recurrent_weight_mask',hh)
        with torch.no_grad():
            self.lstm.weight_ih_l0.mul_(self.input_weight_mask)
            self.lstm.weight_hh_l0.mul_(self.recurrent_weight_mask)
        self.lstm.weight_ih_l0.register_hook(self._mask_input_gradient)
        self.lstm.weight_hh_l0.register_hook(self._mask_recurrent_gradient)
    def _mask_input_gradient(self,gradient):return gradient*self.input_weight_mask
    def _mask_recurrent_gradient(self,gradient):return gradient*self.recurrent_weight_mask
    def assert_structural_zeros(self):
        assert torch.count_nonzero(self.lstm.weight_ih_l0[self.input_weight_mask==0]).item()==0
        assert torch.count_nonzero(self.lstm.weight_hh_l0[self.recurrent_weight_mask==0]).item()==0
    def effective_parameter_count(self):
        return int(self.input_weight_mask.sum().item()+self.recurrent_weight_mask.sum().item()+sum(p.numel() for n,p in self.named_parameters() if n not in ['lstm.weight_ih_l0','lstm.weight_hh_l0']))
    def forward(self,x):
        assert x.shape[-1]==27
        z,_=self.lstm(x)
        return self.head(self.output_dropout(z[:,-1])).squeeze(-1)

def reference_ea_forward(model,x):
    """Independent direct EA recurrence, with no nn.LSTM call."""
    hsize=model.lstm.hidden_size
    w=model.lstm.weight_ih_l0*model.input_weight_mask
    u=model.lstm.weight_hh_l0*model.recurrent_weight_mask
    b=model.lstm.bias_ih_l0+model.lstm.bias_hh_l0
    statics=x[:,0,7:]
    gate_i=torch.sigmoid(statics@w[:hsize,7:].T+b[:hsize])
    h=x.new_zeros((x.shape[0],hsize));c=torch.zeros_like(h)
    for t in range(x.shape[1]):
        fgo=x[:,t,:7]@w[hsize:,:7].T+h@u[hsize:,:].T+b[hsize:]
        f,g,o=fgo.chunk(3,dim=-1)
        c=torch.sigmoid(f)*c+gate_i*torch.tanh(g)
        h=torch.sigmoid(o)*torch.tanh(c)
    return model.head(h).squeeze(-1)

def verify_ea_architecture():
    state=(random.getstate(),np.random.get_state(),torch.get_rng_state(),torch.cuda.get_rng_state_all())
    original_tf32=torch.backends.cudnn.allow_tf32
    original_matmul_tf32=torch.backends.cuda.matmul.allow_tf32
    try:
        torch.manual_seed(123)
        a=MeteorologyLSTM(hidden=4).double().eval()
        b=MeteorologyLSTM(hidden=4).double().eval();b.load_state_dict(a.state_dict())
        d=torch.randn(2,9,7,dtype=torch.float64)
        s=torch.randn(2,20,dtype=torch.float64)
        x=torch.cat([d,s[:,None,:].expand(-1,9,-1)],-1)
        qa=a(x);qb=reference_ea_forward(b,x)
        assert torch.allclose(qa,qb,rtol=1e-10,atol=1e-10)
        qa.square().sum().backward();qb.square().sum().backward()
        for (_,pa),(_,pb) in zip(a.named_parameters(),b.named_parameters()):
            assert torch.allclose(pa.grad,pb.grad,rtol=1e-9,atol=1e-10)
        opt_test=torch.optim.Adam(a.parameters(),lr=.001);opt_test.step();a.assert_structural_zeros()
        # Exact separation of static input gate and dynamic first-step gates.
        w=a.lstm.weight_ih_l0;u=a.lstm.weight_hh_l0;bias=a.lstm.bias_ih_l0+a.lstm.bias_hh_l0
        altered=x.clone();altered[:,:,:7]+=3
        ia=x[:,0]@w[:4].T+bias[:4];ib=altered[:,0]@w[:4].T+bias[:4]
        assert torch.equal(ia,ib) and torch.count_nonzero(u[:4]).item()==0
        altered=x.clone();altered[:,:,7:]+=3
        assert torch.equal(x[:,0]@w[4:].T,altered[:,0]@w[4:].T)
        # Compare full FP32 calculations; restore the baseline TF32 policy after verification.
        torch.backends.cudnn.allow_tf32=False
        torch.backends.cuda.matmul.allow_tf32=False
        # CUDA/cuDNN outputs and gradients agree with the direct equations too.
        c=MeteorologyLSTM(hidden=4).to(DEVICE).train();c.output_dropout.eval()
        e=MeteorologyLSTM(hidden=4).to(DEVICE).train();e.output_dropout.eval();e.load_state_dict(c.state_dict())
        xx=x.float().to(DEVICE);q1=c(xx);q2=reference_ea_forward(e,xx)
        assert torch.allclose(q1,q2,rtol=2e-5,atol=2e-6)
        q1.square().sum().backward();q2.square().sum().backward()
        for (name,pa),(_,pb) in zip(c.named_parameters(),e.named_parameters()):
            error=(pa.grad-pb.grad).abs().max().item()
            print("GPU GRAD CHECK",name,"max_abs_error",error,"reference_max",pb.grad.abs().max().item(),flush=True)
            assert torch.allclose(pa.grad,pb.grad,rtol=2e-4,atol=2e-6),name
        return dict(status='passed',cpu_forward_and_gradient_equivalence=True,cuda_forward_and_gradient_equivalence=True,static_dynamic_gate_separation=True,masked_weights_remain_zero_after_adam=True)
    finally:
        torch.backends.cudnn.allow_tf32=original_tf32
        torch.backends.cuda.matmul.allow_tf32=original_matmul_tf32
        random.setstate(state[0]);np.random.set_state(state[1]);torch.set_rng_state(state[2]);torch.cuda.set_rng_state_all(state[3])

def progress(**info):
    import html
    info.update(updated_utc=time.strftime('%Y-%m-%d %H:%M:%S UTC',time.gmtime()))
    atomic(OUT/'status.json',info)
    rows=globals().get('history',[])
    text='<html><head><meta charset="utf-8"></head><body style="font:16px sans-serif;margin:26px"><h1>EA-LSTM Daymet + PCA10 mean/std · H256 / dropout 0.2 / adaptive 50</h1><p>Refresh this page for updates. Test performance is calculated only after checkpoint selection.</p><pre>'+html.escape(json.dumps(info,indent=2))+'</pre><h2>Completed epochs</h2><table border="1" cellpadding="7"><tr><th>Epoch</th><th>Training loss</th><th>Validation loss</th><th>LR used</th><th>Next LR</th><th>Seconds</th></tr>'
    for r in rows:text+='<tr>'+''.join('<td>'+str(r.get(k,''))+'</td>' for k in ['epoch','training_loss','validation_loss','learning_rate_used','learning_rate_next','epoch_seconds'])+'</tr>'
    text+='</table></body></html>'
    (OUTPUT_BASE/'progress.html').write_text(text)

def checkpoint(path,model,opt,epoch,best,history):
    model.assert_structural_zeros()
    state=dict(model=model.state_dict(),optimizer=opt.state_dict(),epoch=epoch,best_validation=best,best_epoch=best_epoch,history=history,lr_bad_epochs=lr_bad_epochs,early_bad_epochs=early_bad_epochs,training_complete=training_complete,stop_reason=stop_reason,experiment_hash=DIGEST,python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all())
    tmp=path.with_suffix('.tmp');torch.save(state,tmp);tmp.replace(path)

def validate(model):
    model.eval();sums=np.zeros(1326);nums=np.zeros(1326)
    with torch.no_grad():
        for x,y,bi,t in val_loader:
            q=model(x.to(DEVICE,non_blocking=True)).cpu().numpy();b=bi.numpy();err=(q-y.numpy())**2/p['loss_scales'][b]**2
            sums+=np.bincount(b,weights=err,minlength=1326);nums+=np.bincount(b,minlength=1326)
    return float(np.mean(sums[nums>0]/nums[nums>0])),int((nums>0).sum())

atomic(OUTPUT_BASE/'architecture_verification.json',verify_ea_architecture())
print('EA-LSTM forward/backward and gate-separation checks passed',flush=True)


GPU GRAD CHECK lstm.weight_ih_l0 max_abs_error 4.731118679046631e-07 reference_max 0.12517330050468445


GPU GRAD CHECK lstm.weight_hh_l0 max_abs_error 6.612390279769897e-08 reference_max 0.01262090727686882


GPU GRAD CHECK lstm.bias_ih_l0 max_abs_error 1.341104507446289e-07 reference_max 0.1460246592760086


GPU GRAD CHECK lstm.bias_hh_l0 max_abs_error 1.341104507446289e-07 reference_max 0.1460246592760086


GPU GRAD CHECK head.weight max_abs_error 1.2218952178955078e-06 reference_max 0.1932663470506668


GPU GRAD CHECK head.bias max_abs_error 1.1920928955078125e-06 reference_max 1.5708401203155518


EA-LSTM forward/backward and gate-separation checks passed


## 4. Train and validate
Full-batch GPU check, epoch checkpoints, adaptive LR and validation-only selection.

In [5]:
model=MeteorologyLSTM().to(DEVICE);opt=torch.optim.Adam(model.parameters(),lr=CFG['learning_rate'])
assert model.lstm.input_size==27 and model.lstm.hidden_size==256 and model.lstm.num_layers==1
assert model.output_dropout.p==0.2 and sum(v.numel() for v in model.parameters())==292097
assert model.effective_parameter_count()==209409
model.assert_structural_zeros()
# Preserve RNG state around the batch-memory check; the check never updates parameters.
saved_rng=(random.getstate(),np.random.get_state(),torch.get_rng_state(),torch.cuda.get_rng_state_all())
torch.cuda.reset_peak_memory_stats();model.train();x,y,bi,t=next(iter(loader))
assert len(x)==128
q=model(x.to(DEVICE));scale=torch.tensor(p['loss_scales'][bi.numpy()],dtype=torch.float32,device=DEVICE)
loss=(((q-y.to(DEVICE))/scale)**2).mean();loss.backward();assert torch.isfinite(loss);opt.zero_grad(set_to_none=True)
train_memory=torch.cuda.max_memory_allocated();model.eval()
with torch.no_grad():
    vx,vy,vbi,vt=next(iter(val_loader));assert len(vx)==256
    vq=model(vx.to(DEVICE));assert torch.isfinite(vq).all()
torch.cuda.synchronize()
atomic(OUTPUT_BASE/'gpu_memory_check.json',dict(status='passed',gpu=torch.cuda.get_device_name(0),training_batch=128,validation_batch=256,train_peak_allocated_bytes=train_memory,total_peak_allocated_bytes=torch.cuda.max_memory_allocated(),device_total_bytes=torch.cuda.get_device_properties(0).total_memory,precision='float32',parameters=292097,effective_parameters=209409,structural_zero_parameters=82688))
del x,y,bi,t,q,scale,loss,vx,vy,vbi,vt,vq
random.setstate(saved_rng[0]);np.random.set_state(saved_rng[1]);torch.set_rng_state(saved_rng[2]);torch.cuda.set_rng_state_all(saved_rng[3])
first=0;best=float('inf');best_epoch=0;history=[];lr_bad_epochs=0;early_bad_epochs=0;training_complete=False;stop_reason=''
if (OUT/'last.pt').exists():
    state=torch.load(OUT/'last.pt',map_location='cpu',weights_only=False);assert state['experiment_hash']==DIGEST,'Config or dataset changed; use a new output folder.'
    model.load_state_dict(state['model']);model.assert_structural_zeros();opt.load_state_dict(state['optimizer']);first=state['epoch'];best=state['best_validation'];best_epoch=state['best_epoch'];history=state['history'];lr_bad_epochs=state['lr_bad_epochs'];early_bad_epochs=state['early_bad_epochs'];training_complete=state['training_complete'];stop_reason=state['stop_reason']
    random.setstate(state['python_rng']);np.random.set_state(state['numpy_rng']);torch.set_rng_state(state['torch_rng'].cpu());torch.cuda.set_rng_state_all(state['cuda_rng'])
    print('RESUMED completed epoch',first,'best',best_epoch,'LR',opt.param_groups[0]['lr'],flush=True)
atomic(OUT/'run_config.json',dict(config=CFG,csv_sha256=CSV_HASH,model_inputs=FEATURES,static_features=STATIC_FEATURES,parameters=sum(v.numel() for v in model.parameters()),effective_parameters=model.effective_parameter_count(),architecture='EA-LSTM masked cuDNN',torch_version=torch.__version__,numpy_version=np.__version__,gpu=torch.cuda.get_device_name(0)))
started=time.time();prior_seconds=history[-1]['elapsed_seconds'] if history else 0
if not training_complete:
    for epoch in range(first,CFG['epochs']):
        epoch_start=time.time();lr_used=opt.param_groups[0]['lr'];model.train();sampler.epoch=epoch;total=0.;count=0
        for step,(x,y,bi,t) in enumerate(loader,1):
            opt.zero_grad(set_to_none=True);q=model(x.to(DEVICE,non_blocking=True));target=y.to(DEVICE,non_blocking=True)
            scale=torch.tensor(p['loss_scales'][bi.numpy()],dtype=torch.float32,device=DEVICE);loss=(((q-target)/scale)**2).mean();assert torch.isfinite(loss)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step();total+=float(loss.detach())*len(y);count+=len(y)
            if step%200==0:
                progress(phase='training',epoch=epoch+1,batch=step,batches=len(loader),last_completed_epoch=epoch,best_epoch=best_epoch,learning_rate=lr_used)
                print('PROGRESS',epoch+1,step,'/',len(loader),'seconds',round(time.time()-epoch_start),flush=True)
        v,nval=validate(model);assert math.isfinite(v)
        improved=v<best
        if improved:best=v;best_epoch=epoch+1;lr_bad_epochs=0;early_bad_epochs=0
        else:lr_bad_epochs+=1;early_bad_epochs+=1
        reduced=False
        if lr_bad_epochs>=CFG['lr_patience']:
            for pg in opt.param_groups:pg['lr']=max(CFG['min_lr'],pg['lr']*CFG['lr_factor'])
            reduced=opt.param_groups[0]['lr']<lr_used;lr_bad_epochs=0
        training_complete=early_bad_epochs>=CFG['early_stopping_patience'] or epoch+1>=CFG['epochs']
        stop_reason='early_stopping' if early_bad_epochs>=CFG['early_stopping_patience'] else ('max_epochs' if epoch+1>=CFG['epochs'] else '')
        history.append(dict(epoch=epoch+1,training_loss=total/count,validation_loss=v,validation_donors=nval,learning_rate_used=lr_used,learning_rate_next=opt.param_groups[0]['lr'],learning_rate_reduced=reduced,improved=improved,best_epoch=best_epoch,early_bad_epochs=early_bad_epochs,lr_bad_epochs=lr_bad_epochs,epoch_seconds=round(time.time()-epoch_start,2),elapsed_seconds=round(prior_seconds+time.time()-started,2)))
        checkpoint(OUT/'last.pt',model,opt,epoch+1,best,history)
        if improved:checkpoint(OUT/'best.pt',model,opt,epoch+1,best,history)
        atomic(OUT/'history.json',history);progress(phase='training_complete' if training_complete else 'epoch_complete',epoch=epoch+1,best_epoch=best_epoch,best_validation_loss=best,learning_rate=opt.param_groups[0]['lr'],stop_reason=stop_reason)
        print('EPOCH',history[-1],flush=True)
        if training_complete:break
        if (OUTPUT_BASE/'STOP_AFTER_EPOCH').exists():raise RuntimeError('Operational stop requested after a saved epoch. Remove STOP_AFTER_EPOCH and rerun to resume; no test evaluation has run.')
ck=torch.load(OUT/'best.pt',map_location='cpu',weights_only=False);model.load_state_dict(ck['model']);model.assert_structural_zeros();model.eval()
assert ck['epoch']==min(history,key=lambda r:r['validation_loss'])['epoch']
atomic(OUT/'selection.json',dict(best_epoch=ck['epoch'],best_validation_loss=ck['best_validation'],completed_epochs=len(history),stop_reason=stop_reason,selection_uses='validation loss only',test_evaluation_started=False))
progress(phase='selected_checkpoint',epoch=len(history),best_epoch=ck['epoch'],stop_reason=stop_reason)
print('SELECTED VALIDATION-BEST EPOCH',ck['epoch'],'of',len(history),flush=True)


PROGRESS 1 200 / 2652 seconds 4


PROGRESS 1 400 / 2652 seconds 9


PROGRESS 1 600 / 2652 seconds 13


PROGRESS 1 800 / 2652 seconds 17


PROGRESS 1 1000 / 2652 seconds 21


PROGRESS 1 1200 / 2652 seconds 26


... (691 outputs left out)

PROGRESS 50 2400 / 2652 seconds 50


PROGRESS 50 2600 / 2652 seconds 54


EPOCH {'epoch': 50, 'training_loss': 0.21451254321619037, 'validation_loss': 0.21933609261436005, 'validation_donors': 1290, 'learning_rate_used': 3.125e-05, 'learning_rate_next': 3.125e-05, 'learning_rate_reduced': False, 'improved': False, 'best_epoch': 46, 'early_bad_epochs': 4, 'lr_bad_epochs': 1, 'epoch_seconds': 65.06, 'elapsed_seconds': 3275.39}


SELECTED VALIDATION-BEST EPOCH 46 of 50


## 5. Test seen and unseen catchments

In [6]:
def scores(y,q):
    good=np.isfinite(y)&np.isfinite(q); y,q=y[good],q[good]
    if len(y)<2 or y.std()==0: return dict(n=len(y),NSE=np.nan,KGE=np.nan,bias_percent=np.nan)
    nse=1-np.sum((q-y)**2)/np.sum((y-y.mean())**2)
    corr=np.corrcoef(y,q)[0,1] if q.std()>0 else np.nan
    kge=1-np.sqrt((corr-1)**2+(q.std()/y.std()-1)**2+(q.mean()/y.mean()-1)**2) if y.mean()!=0 else np.nan
    return dict(n=len(y),NSE=float(nse),KGE=float(kge),bias_percent=float(100*(q.sum()-y.sum())/y.sum()) if y.sum()!=0 else np.nan)

def evaluate(rs,tag):
    folder=OUT/tag;folder.mkdir(exist_ok=True);metrics=[]
    for i,r in enumerate(rs):
        ds=Windows([r],p,'test');ts=[];pred=[]
        with torch.no_grad():
            for x,y,bi,t in DataLoader(ds,batch_size=256,num_workers=0,pin_memory=True):
                q=model(x.to(DEVICE,non_blocking=True)).cpu().numpy()*p['ystd']+p['ymean'];ts.extend(t.numpy().tolist());pred.extend(q.tolist())
        ts=np.array(ts,dtype=int);q=np.array(pred);obs=r['y'][ts]
        assert np.isfinite(q).all() and np.all(r['dates'][ts]>=np.datetime64('2005-01-01')) and np.all(r['dates'][ts]<np.datetime64('2010-01-01'))
        save_csv(folder/(r['id']+'_predictions.csv'),[dict(date=str(r['dates'][t]),observed_mm_day=float(o),predicted_mm_day=float(v)) for t,o,v in zip(ts,obs,q)],['date','observed_mm_day','predicted_mm_day'])
        good=np.isfinite(obs)&np.isfinite(q);error=q[good]-obs[good]
        extra=dict(RMSE=float(np.sqrt(np.mean(error**2))) if len(error) else np.nan,MAE=float(np.mean(np.abs(error))) if len(error) else np.nan)
        metrics.append(dict(basin_id=r['id'],**scores(obs,q),**extra))
        if (i+1)%100==0 or i+1==len(rs):print('TEST',tag,i+1,'/',len(rs),flush=True);atomic(OUT/'status.json',dict(phase='testing',test=tag,completed=i+1,total=len(rs)))
    save_csv(folder/'metrics.csv',metrics);return metrics
unseen_metrics=evaluate(held,'unseen100');temporal_metrics=evaluate(donors,'donor_temporal')
all_metrics=[]
for tag,rows in [('Seen',temporal_metrics),('Unseen',unseen_metrics)]:
    for r in rows:
        r=dict(r,seen_unseen=tag,catchment_split='training_donor' if tag=='Seen' else 'unseen_test',test_start='2005-01-01',test_end='2009-12-31',model='EA-LSTM Daymet + PCA10 mean/std H256 dropout0.2 adaptive50')
        r['status']='scored' if r['n']>=2 and np.isfinite(r['NSE']) else ('no_test_observations' if r['n']==0 else 'insufficient_or_constant_observations')
        all_metrics.append(r)
all_metrics.sort(key=lambda r:r['basin_id'])
assert len(all_metrics)==len({r['basin_id'] for r in all_metrics})==1426
assert sum(r['seen_unseen']=='Seen' for r in all_metrics)==1326
assert sum(r['seen_unseen']=='Unseen' for r in all_metrics)==100
save_csv(OUTPUT_BASE/'basin_model_performance.csv',all_metrics,['basin_id','seen_unseen','catchment_split','model','test_start','test_end','n','NSE','KGE','RMSE','MAE','bias_percent','status'])
summary_rows=[]
for tag in ('Seen','Unseen'):
    rows=[r for r in all_metrics if r['seen_unseen']==tag];scored=[r for r in rows if np.isfinite(r['NSE'])]
    summary_rows.append(dict(seen_unseen=tag,total_basins=len(rows),scored_basins=len(scored),median_NSE=float(np.median([r['NSE'] for r in scored])),median_KGE=float(np.nanmedian([r['KGE'] for r in scored])),median_bias_percent=float(np.nanmedian([r['bias_percent'] for r in scored]))))
save_csv(OUTPUT_BASE/'model_performance_summary.csv',summary_rows)
atomic(OUT/'summary.json',dict(model='EA-LSTM Daymet + PCA10 mean/std H256 dropout0.2 adaptive50',best_epoch=ck['epoch'],test_start='2005-01-01',test_end='2009-12-31',groups=summary_rows))
atomic(OUT/'status.json',dict(phase='complete',basins=1426,seen=1326,unseen=100))
print('Both test groups complete. Basin-level metrics exported beside this notebook.')


TEST unseen100 100 / 100


TEST donor_temporal 100 / 1326


TEST donor_temporal 200 / 1326


TEST donor_temporal 300 / 1326


TEST donor_temporal 400 / 1326


TEST donor_temporal 500 / 1326


... (6 outputs left out)

TEST donor_temporal 1200 / 1326


TEST donor_temporal 1300 / 1326


TEST donor_temporal 1326 / 1326


Both test groups complete. Basin-level metrics exported beside this notebook.


## 6. Verify metrics and compare with the H256 LSTM baseline

In [7]:
import html
baseline_reference=json.loads(zlib.decompress(base64.b64decode('...')))
baseline_index={r['basin_id']:r for r in baseline_reference}
assert set(baseline_index)=={r['basin_id'] for r in all_metrics}
def flow_errors(y,q):
    good=np.isfinite(y)&np.isfinite(q);y=np.asarray(y[good],dtype='float64');q=np.asarray(q[good],dtype='float64');out={}
    out['mean_bias_mm_day']=float((q-y).mean()) if len(y) else np.nan
    out['negative_predictions']=int(np.sum(q<0));out['negative_prediction_percent']=float(np.mean(q<0)*100) if len(q) else np.nan
    for label,quant in [('low',.1),('high',.9)]:
        threshold=float(np.quantile(y,quant)) if len(y) else np.nan
        keep=y<=threshold if label=='low' else y>=threshold
        a,c=y[keep],q[keep];e=c-a
        out.update({label+'_threshold_mm_day':threshold,label+'_n':len(a),label+'_RMSE':float(np.sqrt(np.mean(e*e))) if len(a) else np.nan,label+'_MAE':float(np.mean(np.abs(e))) if len(a) else np.nan,label+'_bias_mm_day':float(e.mean()) if len(a) else np.nan,label+'_bias_percent':float(100*e.sum()/a.sum()) if len(a) and a.sum()!=0 else np.nan})
    return out

verified=0
for row in all_metrics:
    tag='donor_temporal' if row['seen_unseen']=='Seen' else 'unseen100'
    daily=list(csv.DictReader((OUT/tag/(row['basin_id']+'_predictions.csv')).open()))
    dates=[x['date'] for x in daily];assert len(dates)==len(set(dates)) and all('2005-01-01'<=d<='2009-12-31' for d in dates)
    date_hash=hashlib.sha256('\n'.join(dates).encode()).hexdigest();assert date_hash==baseline_index[row['basin_id']]['test_dates_sha256']
    y=np.array([float(x['observed_mm_day']) for x in daily],dtype='float32');q=np.array([float(x['predicted_mm_day']) for x in daily],dtype='float64')
    assert np.isfinite(q).all();good=np.isfinite(y);a,c=y[good],q[good];e=c-a
    values=dict(NSE=np.nan,KGE=np.nan,bias_percent=np.nan,RMSE=float(np.sqrt(np.mean(e*e))) if len(a) else np.nan,MAE=float(np.mean(np.abs(e))) if len(a) else np.nan)
    if len(a)>=2 and a.std()>0:
        values['NSE']=float(1-np.sum(e*e)/np.sum((a-a.mean())**2))
        correlation=np.corrcoef(a,c)[0,1] if c.std()>0 else np.nan
        values['KGE']=float(1-np.sqrt((correlation-1)**2+(c.std()/a.std()-1)**2+(c.mean()/a.mean()-1)**2)) if a.mean()!=0 else np.nan
        values['bias_percent']=float(100*(c.sum()-a.sum())/a.sum()) if a.sum()!=0 else np.nan
    assert len(a)==row['n']==int(baseline_index[row['basin_id']]['n'])
    for k,v in values.items():assert np.allclose(v,row[k],rtol=2e-5,atol=2e-5,equal_nan=True),(row['basin_id'],k)
    row.update(flow_errors(y,q));row['test_dates_sha256']=date_hash
    for k in ['low_threshold_mm_day','high_threshold_mm_day','low_n','high_n']:
        assert np.allclose(float(row[k]),float(baseline_index[row['basin_id']][k]),rtol=1e-8,atol=1e-8,equal_nan=True),(row['basin_id'],k)
    verified+=1
    if verified%200==0:print('INDEPENDENT METRICS VERIFIED',verified,flush=True)
save_csv(OUTPUT_BASE/'basin_model_performance.csv',all_metrics)
save_csv(OUTPUT_BASE/'baseline_seed42_basin_performance.csv',baseline_reference)
metrics=['NSE','KGE','RMSE','MAE','bias_percent','mean_bias_mm_day','low_RMSE','low_MAE','low_bias_mm_day','low_bias_percent','high_RMSE','high_MAE','high_bias_mm_day','high_bias_percent']
summaries=[];paired=[];basin_comparison=[]
def number(r,k):
    try:return float(r[k])
    except (ValueError,TypeError):return np.nan
for group in ['Seen','Unseen']:
    for model_name,rows in [('Baseline LSTM H256 / dropout 0.2 / adaptive50',baseline_reference),('EA-LSTM H256 / dropout 0.2 / adaptive50',all_metrics)]:
        rs=[r for r in rows if r['seen_unseen']==group]
        for metric in metrics:
            a=np.array([number(r,metric) for r in rs]);a=a[np.isfinite(a)]
            summaries.append(dict(model=model_name,group=group,metric=metric,total_basins=len(rs),n=len(a),mean=float(a.mean()) if len(a) else np.nan,median=float(np.median(a)) if len(a) else np.nan,q1=float(np.quantile(a,.25)) if len(a) else np.nan,q3=float(np.quantile(a,.75)) if len(a) else np.nan,std=float(a.std(ddof=1)) if len(a)>1 else np.nan,minimum=float(a.min()) if len(a) else np.nan,maximum=float(a.max()) if len(a) else np.nan,negative=int(np.sum(a<0)) if metric in ['NSE','KGE'] else ''))
    for metric in ['NSE','KGE','RMSE','MAE','low_RMSE','low_MAE','high_RMSE','high_MAE']:
        pairs=[(number(baseline_index[r['basin_id']],metric),number(r,metric)) for r in all_metrics if r['seen_unseen']==group]
        pairs=np.array(pairs);pairs=pairs[np.isfinite(pairs).all(1)];d=pairs[:,1]-pairs[:,0]
        higher=metric in ['NSE','KGE'];win=d>0 if higher else d<0;lose=d<0 if higher else d>0
        paired.append(dict(group=group,metric=metric,n=len(d),baseline_median=float(np.median(pairs[:,0])),new_median=float(np.median(pairs[:,1])),median_paired_change=float(np.median(d)),mean_paired_change=float(d.mean()),improved=int(win.sum()),worsened=int(lose.sum()),tied=int((d==0).sum()),better_direction='higher' if higher else 'lower'))
for r in all_metrics:
    old=baseline_index[r['basin_id']];row=dict(basin_id=r['basin_id'],seen_unseen=r['seen_unseen'],n=r['n'])
    for metric in metrics:
        a,b=number(old,metric),number(r,metric);row['baseline_'+metric]=a;row['new_'+metric]=b;row['change_'+metric]=b-a
    basin_comparison.append(row)
save_csv(OUTPUT_BASE/'model_performance_summary.csv',summaries)
save_csv(OUTPUT_BASE/'paired_comparison_summary.csv',paired)
save_csv(OUTPUT_BASE/'basin_baseline_comparison.csv',basin_comparison)
history=json.loads((OUT/'history.json').read_text());selection=json.loads((OUT/'selection.json').read_text())
assert history and len(history)<=50 and [r['epoch'] for r in history]==list(range(1,len(history)+1))
assert selection['best_epoch']==min(history,key=lambda r:r['validation_loss'])['epoch']
best0=float('inf');bad=0;lr_bad=0;rate=.001
for row in history:
    assert row['learning_rate_used']==rate
    if row['validation_loss']<best0:best0=row['validation_loss'];bad=0;lr_bad=0
    else:bad+=1;lr_bad+=1
    if lr_bad>=3:rate=max(.00001,rate*.5);lr_bad=0
    assert row['learning_rate_next']==rate and row['early_bad_epochs']==bad and row['lr_bad_epochs']==lr_bad
assert (len(history)==50 and selection['stop_reason']=='max_epochs') or (bad==10 and selection['stop_reason']=='early_stopping')
audit=json.loads((OUTPUT_BASE/'protocol_verification.json').read_text())
for path,expected in audit['preserved_baseline_sha256'].items():
    pth=Path(path)
    if pth.exists():
        hh=hashlib.sha256()
        with pth.open('rb') as f:
            for part in iter(lambda:f.read(16*1024*1024),b''):hh.update(part)
        assert hh.hexdigest()==expected,'Prior experiment changed: '+path
def table(rows):
    if not rows:return '<p>No rows.</p>'
    keys=list(rows[0])
    def fmt(v):
        if isinstance(v,(float,np.floating)):return f'{v:.4f}' if np.isfinite(v) else '—'
        return str(v)
    return '<div style="overflow:auto"><table><tr>'+''.join('<th>'+html.escape(k.replace('_',' '))+'</th>' for k in keys)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(fmt(r[k]))+'</td>' for k in keys)+'</tr>' for r in rows)+'</table></div>'
def loss_chart(rows):
    width=850;height=280;left=65;top=20;pw=750;ph=210
    vals=[r[k] for r in rows for k in ['training_loss','validation_loss']];low=min(vals);high=max(vals);span=max(high-low,1e-8)
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" role="img" aria-label="Training and validation loss by epoch"><rect width="100%" height="100%" fill="white"/>']
    for k,col in [('training_loss','#087f83'),('validation_loss','#d58536')]:
        pts=' '.join(f'{left+pw*i/max(1,len(rows)-1):.2f},{top+ph*(1-(r[k]-low)/span):.2f}' for i,r in enumerate(rows));parts.append(f'<polyline points="{pts}" fill="none" stroke="{col}" stroke-width="2"/>')
    parts.append(f'<text x="{left}" y="265" font-size="13">Epoch 1</text><text x="740" y="265" font-size="13">Epoch {len(rows)}</text><text x="5" y="30" font-size="12">{high:.3g}</text><text x="5" y="230" font-size="12">{low:.3g}</text></svg>')
    return ''.join(parts)
chart=loss_chart(history);(OUTPUT_BASE/'training_validation_loss.svg').write_text(chart)
headline=[r for r in paired if r['metric'] in ['NSE','KGE','low_RMSE','high_RMSE']]
report='<html><head><meta charset="utf-8"><style>body{font:15px sans-serif;max-width:1400px;margin:28px;color:#173c42}td,th{padding:8px;border:1px solid #dbe5e4;text-align:right}table{border-collapse:collapse}th{background:#eaf3f1}h1,h2{color:#126b70}svg{max-width:950px}</style></head><body><h1>EA-LSTM Daymet + PCA10 mean/std · H256 / dropout 0.2 / adaptive50</h1><p>Seed 42. Train 1980–1999; validation 2000–2004; test 2005–2009. Completed '+str(len(history))+' epochs; selected epoch '+str(selection['best_epoch'])+' by validation loss only. Stop reason: '+selection['stop_reason']+'.</p><h2>Paired comparison with the H256 LSTM seed-42 baseline</h2>'+table(headline)+'<p>Changes are new minus baseline. Higher NSE/KGE is better; lower RMSE/MAE is better. Paired comparisons use only basins with finite scores in both experiments.</p><h2>Score distributions and negative-score counts</h2>'+table([r for r in summaries if r['metric'] in ['NSE','KGE','RMSE','bias_percent']])+'<h2>Training and validation</h2><p>Teal: training loss; orange: validation loss. Loss uses the unchanged basin-scaled squared-error definition.</p>'+chart+'<h2>Low-flow and high-flow evaluation</h2><p>Within each basin, low flow means observed test discharge ≤ Q10; high flow means ≥ Q90, including ties. Quantiles use finite observed/predicted test pairs only after checkpoint selection, and match the baseline exactly. Thresholds, sample counts, absolute and percentage errors are in the per-basin CSV. Percent bias is undefined when observed subset sum is zero.</p>'+table([r for r in paired if r['metric'] in ['low_RMSE','low_MAE','high_RMSE','high_MAE']])+'<h2>Saved results</h2><p>basin_model_performance.csv: all 1,426 basins, Seen/Unseen labels and extended flow errors. basin_baseline_comparison.csv: paired basin-level metrics. model_performance_summary.csv: distribution statistics. paired_comparison_summary.csv: improvement counts. results_seed42/: daily predictions, checkpoints, preprocessing, history and selection.</p><p>All daily metrics were independently recalculated. Original files were preserved. This is one seed comparing EA gating with the H256 LSTM under the same training protocol. Active parameter counts differ. This does not establish repeatable superiority.</p></body></html>'
(OUTPUT_BASE/'model_report.html').write_text(report)
atomic(OUTPUT_BASE/'results_verification.json',dict(status='passed',basins=verified,all_daily_metrics_recomputed=True,test_dates_and_thresholds_match_baseline=True,scheduler_and_stop_rules_verified=True,prior_experiment_hashes_preserved=True))
completion=dict(status='complete',model='EA-LSTM Daymet + PCA10 mean/std H256 dropout0.2 adaptive50',seed=42,epochs=len(history),best_epoch=selection['best_epoch'],stop_reason=selection['stop_reason'],basins=1426,seen=1326,unseen=100,parameters=292097,effective_parameters=209409,structural_zero_parameters=82688)
atomic(OUTPUT_BASE/'completion.json',completion);progress(phase='complete',**completion)
display(HTML('<h2>Paired baseline comparison</h2>'+table(headline)))
display(HTML('<h2>Performance distributions</h2>'+table([r for r in summaries if r['metric'] in ['NSE','KGE']])))
display(HTML('<h2>Training history</h2>'+chart))
print('COMPLETE',json.dumps(completion),flush=True)


INDEPENDENT METRICS VERIFIED 200


INDEPENDENT METRICS VERIFIED 400


INDEPENDENT METRICS VERIFIED 600


INDEPENDENT METRICS VERIFIED 800


INDEPENDENT METRICS VERIFIED 1000


INDEPENDENT METRICS VERIFIED 1200


INDEPENDENT METRICS VERIFIED 1400


group,metric,n,baseline median,new median,median paired change,mean paired change,improved,worsened,tied,better direction
Seen,NSE,1317,0.7399,0.7162,-0.0184,-0.0351,478,839,0,higher
Seen,KGE,1317,0.7234,0.7127,-0.0170,-0.0320,542,775,0,higher
Seen,low_RMSE,1317,0.0887,0.1089,0.0113,0.0060,464,853,0,lower
Seen,high_RMSE,1317,2.2209,2.3289,0.0487,0.2057,523,794,0,lower
Unseen,NSE,100,0.6078,0.6305,0.0291,0.0609,63,37,0,higher
Unseen,KGE,100,0.5636,0.5822,0.0231,0.0257,54,46,0,higher
Unseen,low_RMSE,100,0.1056,0.1170,0.0015,-0.0218,49,51,0,lower
Unseen,high_RMSE,100,2.5390,2.7732,-0.0929,-0.2246,62,38,0,lower


model,group,metric,total basins,n,mean,median,q1,q3,std,minimum,maximum,negative
Baseline LSTM H256 / dropout 0.2 / adaptive50,Seen,NSE,1326,1317,0.6216,0.7399,0.5885,0.8316,0.7719,-16.4451,0.9601,45
Baseline LSTM H256 / dropout 0.2 / adaptive50,Seen,KGE,1326,1317,0.5995,0.7234,0.5450,0.8337,0.8356,-25.4145,0.9694,56
EA-LSTM H256 / dropout 0.2 / adaptive50,Seen,NSE,1326,1317,0.5866,0.7162,0.5646,0.8074,0.7221,-13.9869,0.9577,56
EA-LSTM H256 / dropout 0.2 / adaptive50,Seen,KGE,1326,1317,0.5675,0.7127,0.5355,0.8146,0.6152,-9.5410,0.9671,76
Baseline LSTM H256 / dropout 0.2 / adaptive50,Unseen,NSE,100,100,0.4559,0.6078,0.4147,0.7559,0.5474,-2.2005,0.9456,10
Baseline LSTM H256 / dropout 0.2 / adaptive50,Unseen,KGE,100,100,0.4405,0.5636,0.2782,0.7316,0.4161,-1.2303,0.9365,13
EA-LSTM H256 / dropout 0.2 / adaptive50,Unseen,NSE,100,100,0.5168,0.6305,0.4502,0.7710,0.4289,-1.9185,0.9314,10
EA-LSTM H256 / dropout 0.2 / adaptive50,Unseen,KGE,100,100,0.4662,0.5822,0.3373,0.7380,0.3857,-1.0656,0.9367,17


COMPLETE {"status": "complete", "model": "EA-LSTM Daymet + PCA10 mean/std H256 dropout0.2 adaptive50", "seed": 42, "epochs": 50, "best_epoch": 46, "stop_reason": "max_epochs", "basins": 1426, "seen": 1326, "unseen": 100, "parameters": 292097, "effective_parameters": 209409, "structural_zero_parameters": 82688}
